# Exploratory analysis of the VGChartz 2024 extract

This notebook describes the forecasting target, its coverage over release years, missing values and right-censoring. These properties motivate the design recorded in `ROADMAP.md`: only rows with observed sales are modelled, the target is modelled as $z = \log(1 + y)$, and validation is by release year.

All computations call functions of the `vgsales` package. The notebook reads `data/raw/vgchartz-2024.csv` (acquisition steps in `data/README.md`). When the file is absent it falls back to the seeded synthetic generator; tables and figures then describe the generator rather than the market and are labelled "synthetic". Figures are written to `reports/figures/` only for the real extract.

In [ ]:
# Setup
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from vgsales.config import DEFAULT_CONFIG_PATH, load_config
from vgsales.data.cleaning import clean
from vgsales.data.loading import coerce_raw, load_raw
from vgsales.data.profiling import (
    censoring_summary,
    coverage_by_year,
    missingness,
    target_summary,
    top_share,
)
from vgsales.data.synthetic import SyntheticSpec, make_synthetic
from vgsales.data.temporal import apply_release_scope

ROOT = DEFAULT_CONFIG_PATH.parents[1]
FIGURES = ROOT / "reports" / "figures"
config = load_config()
pd.set_option("display.precision", 3)
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False})

In [ ]:
# Real extract when available, synthetic stand-in otherwise
raw_path = ROOT / config.data.raw_path
if raw_path.is_file():
    loaded = load_raw(raw_path)
    SOURCE = "VGChartz 2024"
else:
    synthetic = make_synthetic(SyntheticSpec(n_titles=3000), seed=config.seed)
    loaded, _ = coerce_raw(synthetic)
    SOURCE = "synthetic"
print(f"source: {SOURCE}; rows: {len(loaded):,}")


def save_figure(fig: plt.Figure, name: str) -> None:
    # Figures of the synthetic stand-in are never written to reports/.
    if SOURCE != "synthetic":
        FIGURES.mkdir(parents=True, exist_ok=True)
        fig.savefig(FIGURES / f"{name}.png", dpi=150, bbox_inches="tight")

## Missing values

Missingness is measured on the loaded extract before cleaning. A missing `total_sales` means that the title was not tracked. Such rows are removed during cleaning and never imputed with zero (known issue 3). The regional columns and `last_update` are listed for completeness only; they are never model inputs.

In [ ]:
# Share of missing values per column, before cleaning
missing = missingness(loaded)
missing

In [ ]:
# Missing share by column
fig, ax = plt.subplots(figsize=(7, 4))
shares = missing["share_missing"].sort_values()
ax.barh(shares.index, shares.to_numpy(), color="tab:blue", height=0.6)
ax.set_xlim(0, 1)
ax.set_xlabel("share of rows with a missing value")
ax.set_title(f"Missing values by column ({SOURCE})")
ax.grid(axis="x", alpha=0.3)
save_figure(fig, "eda_missingness")

## Coverage by release year

`apply_release_scope` places rows on the time axis. It removes rows without a release date, rows outside the configured year range and rows released after the snapshot, and flags releases within `data.censoring_window_years` of the snapshot. Coverage is computed on all scoped rows, tracked or not, so that the tracked share per year is visible. This share matters for two design choices: the first training window must contain enough tracked rows (`split.min_train_years`), and the test set consists of the latest `split.holdout_years` release years.

In [ ]:
# Release-year scope on all rows, tracked or not
all_rows, scope_report = apply_release_scope(loaded, config.data)
print(scope_report.to_dict())
coverage = coverage_by_year(all_rows)
coverage.tail(10)

In [ ]:
# Rows and tracked share per release year, on separate axes
fig, (ax_rows, ax_share) = plt.subplots(
    2, 1, figsize=(9, 6), sharex=True, height_ratios=(2, 1)
)
years = coverage.index.to_numpy()
ax_rows.bar(years, coverage["n_rows"], color="0.8", width=0.8, label="all rows")
ax_rows.bar(years, coverage["n_tracked"], color="tab:blue", width=0.8, label="tracked")
ax_rows.set_ylabel("rows")
ax_rows.legend(frameon=False, ncol=2, loc="lower right", bbox_to_anchor=(1, 1))
ax_rows.set_title(f"Rows by release year ({SOURCE})", loc="left")
ax_share.plot(years, coverage["tracked_share"], color="tab:blue", linewidth=2)
ax_share.set_ylim(0, 1)
ax_share.set_ylabel("tracked share")
ax_share.set_xlabel("release year")
for ax in (ax_rows, ax_share):
    ax.grid(axis="y", alpha=0.3)
save_figure(fig, "eda_coverage_by_year")

## Target distribution

Statistics below use cleaned rows only, that is rows with an observed target, within the release scope. For sales $y \ge 0$ in millions of units, the modelling scale is $z = \log(1 + y)$ (`target.transform` in `configs/default.yaml`). The skewness reported is the moment coefficient $m_3 / m_2^{3/2}$. The concentration table gives the share of all units sold by the top 1%, 5% and 10% of rows; a high concentration means that error metrics on the raw scale are dominated by a few titles (known issue 11).

In [ ]:
# Observed target after cleaning and release scope
cleaned, clean_report = clean(loaded)
frame, _ = apply_release_scope(cleaned, config.data)
print(clean_report.to_dict())
target_summary(frame["total_sales"])

In [ ]:
# Share of all units sold by the best-selling rows
pd.Series(
    {f"top {p:.0%}": top_share(frame["total_sales"], p) for p in (0.01, 0.05, 0.10)},
    name="share of units",
)

In [ ]:
# Distribution on the raw and the log1p scale
fig, (ax_raw, ax_log) = plt.subplots(1, 2, figsize=(11, 4))
sales = frame["total_sales"].to_numpy()
ax_raw.hist(sales, bins=60, color="tab:blue")
ax_raw.set_xlabel("total sales (millions of units)")
ax_raw.set_ylabel("rows")
ax_raw.set_title("raw scale")
ax_log.hist(np.log1p(sales), bins=60, color="tab:blue")
ax_log.set_xlabel("log(1 + total sales)")
ax_log.set_title("log1p scale")
for ax in (ax_raw, ax_log):
    ax.grid(axis="y", alpha=0.3)
fig.suptitle(f"Observed total sales ({SOURCE})")
save_figure(fig, "eda_target_distribution")

## Right-censoring

Sales in the extract are cumulative up to the snapshot date. A title released shortly before the snapshot has had less time to accumulate sales, so its observed total is a lower bound of its lifetime sales. These rows are flagged, not removed; evaluation reports them separately. The comparison below contrasts flagged and unflagged rows, and the figure shows the median of $z$ by release year with the censoring window shaded.

In [ ]:
# Observed sales of censored and uncensored releases
censoring_summary(frame)

In [ ]:
# Median log1p sales by release year
by_year = frame.groupby("release_year").agg(
    median_log_sales=("total_sales", lambda s: np.log1p(s).median()),
    censored_share=("censored", "mean"),
)
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(by_year.index, by_year["median_log_sales"], color="tab:blue", linewidth=2)
censored_years = by_year.index[by_year["censored_share"] > 0]
if len(censored_years):
    ax.axvspan(
        censored_years.min() - 0.5,
        censored_years.max() + 0.5,
        color="0.9",
        label="censoring window",
    )
    ax.legend(frameon=False)
ax.set_xlabel("release year")
ax.set_ylabel("median log(1 + total sales)")
ax.set_title(f"Median observed sales by release year ({SOURCE})")
ax.grid(axis="y", alpha=0.3)
save_figure(fig, "eda_censoring")

## Findings

Results to be filled from a full run on the real extract. The points to record are the tracked share per release year and the first year with enough tracked rows for training, the skewness of the target on both scales and the concentration of units, the coverage of `critic_score`, and the size of the difference between censored and uncensored releases.

## Assumptions and limitations

- VGChartz sales are estimates compiled from public sources, not audited figures, and tracking is selective: larger titles are more likely to be tracked. Models fitted on tracked rows describe tracked titles only.
- `last_update` is used only to infer the snapshot date when `data.snapshot_date` is not set. It is never a model input.
- The censoring flag uses a fixed window in calendar years. Titles with long sales tails can remain under-counted beyond that window.